# BBC Burmese scraper (practice for the capstone)

Collects recent BBC Burmese articles: **title, date, body text, photos and captions**.

**Output** (all inside `data/bbc_burmese/`, which `.gitignore` keeps off GitHub):
- `articles.csv`: one row per article (maps to the ARTICLE table)
- `figures.csv`: one row per photo with its caption (maps to the MEDIA table)
- `raw_html/`: the original page of each article, so you can re-extract later without downloading again
- `images/`: downloaded photos, one folder per article

**Rules this notebook follows:** checks robots.txt, waits between requests, skips articles it already has.
BBC text and photos are copyrighted: use them for research only, and don't publish them on GitHub or the project website.

Install first (inside your venv): `pip install requests beautifulsoup4 pandas feedparser`

## Step 1: Settings

In [8]:
import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib import robotparser
from urllib.parse import urljoin, urlparse

import feedparser
import pandas as pd
import requests
from bs4 import BeautifulSoup

HOME_URL = "https://www.bbc.com/burmese"
RSS_URL = "https://feeds.bbci.co.uk/burmese/rss.xml"
MAX_NEW_ARTICLES = 50      # start small; raise to 20-50 once everything works
DELAY_SECONDS = 2         # wait between every request (pages and images)
USER_AGENT = "Mozilla/5.0 (student research project, Parami University)"

DATA_DIR = Path("data/bbc_burmese")
HTML_DIR = DATA_DIR / "raw_html"
IMG_DIR = DATA_DIR / "images"
ARTICLES_CSV = DATA_DIR / "articles.csv"
FIGURES_CSV = DATA_DIR / "figures.csv"
for folder in (HTML_DIR, IMG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

## Step 2: Polite downloading
`allowed()` checks each website's robots.txt (bbc.com for pages, ichef.bbci.co.uk for images). `fetch()` downloads one URL, waits, and returns `None` instead of crashing if something fails.

In [9]:
_robots_cache = {}

def allowed(url):
    """Return True if the site's robots.txt allows us to download this URL."""
    parts = urlparse(url)
    host = f"{parts.scheme}://{parts.netloc}"
    if host not in _robots_cache:
        rp = robotparser.RobotFileParser(host + "/robots.txt")
        try:
            rp.read()
        except Exception:
            rp = None          # robots.txt unreachable: treat as allowed
        _robots_cache[host] = rp
    rp = _robots_cache[host]
    return True if rp is None else rp.can_fetch(USER_AGENT, url)


def fetch(url):
    """Download one URL politely. Returns the response, or None on failure."""
    if not allowed(url):
        print("  robots.txt does not allow:", url)
        return None
    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()
        return response
    except requests.RequestException as error:
        print("  failed:", url, error)
        return None
    finally:
        time.sleep(DELAY_SECONDS)

## Step 3: Find article links
First tries the RSS feed (the easiest and most polite way). If that gives nothing, it collects article links from the home page instead.

In [10]:
ARTICLE_URL = re.compile(r"^https://www\.bbc\.com/burmese/(articles/[a-z0-9]+|[a-z-]+-\d{6,})$")

def clean_url(url):
    return url.split("?")[0].split("#")[0]

def discover_links():
    links = []
    response = fetch(RSS_URL)
    if response is not None:
        feed = feedparser.parse(response.content)
        links = [clean_url(entry.link) for entry in feed.entries if "link" in entry]
        print(f"RSS feed: {len(links)} links")
    if len(links) < MAX_NEW_ARTICLES:
        response = fetch(HOME_URL)
        if response is not None:
            soup = BeautifulSoup(response.text, "html.parser")
            links = [clean_url(urljoin(HOME_URL, a["href"])) for a in soup.find_all("a", href=True)]
            print(f"Home page: {len(links)} links before filtering")
    links = [u for u in links if ARTICLE_URL.match(u)]
    return list(dict.fromkeys(links))   # remove duplicates, keep order

def article_key(url):
    """BBC's own id for the article: the last part of the URL."""
    return urlparse(url).path.rstrip("/").split("/")[-1]

## Step 4: Extract title, date, body and figures
- **Title and dates** come from the page's hidden JSON-LD data first (most reliable), then from meta tags, then from the visible page.
- **Body** = all paragraphs inside `<main>`, except those inside figures, captions, menus or side boxes.
- **Figures** = every `<figure>` that contains an image, with the largest image size, the caption and the alt text.

In [11]:
ARTICLE_TYPES = {"NewsArticle", "ReportageNewsArticle", "Article", "AnalysisNewsArticle"}
SKIP_INSIDE = ["figure", "figcaption", "aside", "nav", "header", "footer"]

def json_ld(soup):
    """Return the article's JSON-LD block (hidden structured data), or {}."""
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except json.JSONDecodeError:
            continue
        items = data.get("@graph", [data]) if isinstance(data, dict) else data
        for item in items if isinstance(items, list) else []:
            if not isinstance(item, dict):
                continue
            types = item.get("@type")
            types = types if isinstance(types, list) else [types]
            if ARTICLE_TYPES & set(types):
                return item
    return {}

def meta(soup, name):
    tag = soup.find("meta", attrs={"property": name}) or soup.find("meta", attrs={"name": name})
    return tag.get("content") if tag else None

def best_image_url(img, page_url):
    """Pick the largest version listed in srcset, else src."""
    srcset = img.get("srcset") or img.get("data-srcset")
    if srcset:
        candidates = []
        for part in srcset.split(","):
            bits = part.strip().split()
            if not bits:
                continue
            width = int(bits[1][:-1]) if len(bits) > 1 and bits[1][:-1].isdigit() else 0
            candidates.append((width, bits[0]))
        if candidates:
            return urljoin(page_url, max(candidates)[1])
    src = img.get("src") or img.get("data-src")
    return urljoin(page_url, src) if src else None

def parse_article(html, url):
    soup = BeautifulSoup(html, "html.parser")
    ld = json_ld(soup)
    main = soup.find("main") or soup

    h1 = main.find("h1")
    time_tag = main.find("time")
    title = ld.get("headline") or meta(soup, "og:title") or (h1.get_text(" ", strip=True) if h1 else None)
    published = (ld.get("datePublished") or meta(soup, "article:published_time")
                 or (time_tag.get("datetime") if time_tag else None))
    modified = ld.get("dateModified") or meta(soup, "article:modified_time")

    paragraphs = [p.get_text(" ", strip=True) for p in main.find_all("p")
                  if not p.find_parent(SKIP_INSIDE)]
    paragraphs = [text for text in paragraphs if text]
    body = "\n\n".join(paragraphs)

    figures = []
    for fig in main.find_all("figure"):
        img = fig.find("img")
        if img is None:          # e.g. a video player, skipped in this version
            continue
        image_url = best_image_url(img, url)
        if not image_url:
            continue
        caption = fig.find("figcaption")
        figures.append({
            "image_url": image_url,
            "original_caption": caption.get_text(" ", strip=True) if caption else None,
            "alt_text": img.get("alt") or None,
        })

    article = {
        "bbc_id": article_key(url),
        "url": url,
        "title_raw": title,
        "published_at": published,
        "modified_at": modified,
        "body_raw": body,
        "paragraph_count": len(paragraphs),
        "image_count": len(figures),
        "content_hash": hashlib.sha256(body.encode("utf-8")).hexdigest() if body else None,
        "scraped_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    return article, figures

## Step 5: Test on ONE article first
Run this and read the output. If the title, date or body is empty, BBC's page structure is different from what Step 4 expects: open the article in Chrome, right-click → Inspect, and adjust Step 4.

In [12]:
links = discover_links()
print(f"{len(links)} article links found")
if links:
    test = fetch(links[0])
    if test is not None:
        test.encoding = "utf-8"
        article, figures = parse_article(test.text, links[0])
        for key, value in article.items():
            print(f"{key:16}: {str(value)[:120]}")
        print(f"\n{len(figures)} figures")
        for fig in figures[:3]:
            print(fig)

RSS feed: 19 links
Home page: 83 links before filtering
26 article links found
bbc_id          : cvgy4xn5nleo
url             : https://www.bbc.com/burmese/articles/cvgy4xn5nleo
title_raw       : မြန်မာ့မြေရှားလုပ်ကွက်တွေထဲက ဖိအားပေးခိုင်းစေခံနေရတဲ့လိင်လုပ်သားစျေးကွက်
published_at    : 2026-10-26T11:34:32.697Z
modified_at     : 2026-10-26T11:34:32.697Z
body_raw        : မြန်မာပြည်မြောက်ပိုင်း၊ ကချင်ပြည်နယ်ထဲက တရုတ်နယ်စပ်မြို့တွေမှာ မြေရှားသတ္တုတူးဖော်တဲ့ လုပ်ကွက်တွေ ရာနဲ့ချီ ပိုတိုးလာတယ်လ
paragraph_count : 73
image_count     : 5
content_hash    : c7eeb299cc52368feb7571e5fd678c305a5d438882bfb1bc4e0f8a36e550ae96
scraped_at      : 2026-10-27T07:01:02+00:00

5 figures
{'image_url': 'https://ichef.bbci.co.uk/ace/ws/800/cpsprodpb/1989/live/64c4eb20-ac3f-11f1-a540-61c3f7fc4e6c.png.webp', 'original_caption': 'ဓာတ်ပုံ ပုံစာ, ကချင်ပြည်နယ် မြောက်ပိုင်းက မြေရှားသတ္တုတူးဖော်နေတဲ့ လုပ်ငန်းခွင် တစ်ခု', 'alt_text': 'ကချင်ပြည်နယ် မြောက်ပိုင်းက မြေရှားသတ္တုတူးဖော်နေတဲ့ လုပ်ငန်းခွင် တစ်ခု'}
{'image_url':

## Step 6: Scrape new articles and download their photos
Safe to run again later: articles already in `articles.csv` are skipped.

In [13]:
def download_image(url, path):
    if path.exists():
        return True
    response = fetch(url)
    if response is None:
        return False
    path.write_bytes(response.content)
    return True

def save_rows(rows, path):
    """Add new rows to a CSV (rewrites the file so Excel can read Burmese text)."""
    if not rows:
        return
    new = pd.DataFrame(rows)
    if path.exists():
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False, encoding="utf-8-sig")

done = set(pd.read_csv(ARTICLES_CSV)["bbc_id"]) if ARTICLES_CSV.exists() else set()
new_articles, new_figures = [], []

for url in links:
    if len(new_articles) >= MAX_NEW_ARTICLES:
        break
    key = article_key(url)
    if key in done:
        continue
    print("Article:", url)
    response = fetch(url)
    if response is None:
        continue
    response.encoding = "utf-8"
    html_path = HTML_DIR / f"{key}.html"
    html_path.write_text(response.text, encoding="utf-8")

    article, figures = parse_article(response.text, url)
    article["raw_html_path"] = str(html_path)

    for number, fig in enumerate(figures, start=1):
        suffix = Path(urlparse(fig["image_url"]).path).suffix or ".jpg"
        local = IMG_DIR / key / f"{number}{suffix}"
        local.parent.mkdir(parents=True, exist_ok=True)
        ok = download_image(fig["image_url"], local)
        fig.update(bbc_id=key, figure_no=number, local_path=str(local) if ok else None)

    new_articles.append(article)
    new_figures.extend(figures)
    print(f"  saved: {article['paragraph_count']} paragraphs, {len(figures)} figures")

save_rows(new_articles, ARTICLES_CSV)
save_rows(new_figures, FIGURES_CSV)
print(f"\nDone: {len(new_articles)} new articles, {len(new_figures)} new figures")

Article: https://www.bbc.com/burmese/articles/cwp93m03nkdko
  saved: 47 paragraphs, 21 figures
Article: https://www.bbc.com/burmese/articles/cke8rq7eng6mo
  saved: 44 paragraphs, 4 figures
Article: https://www.bbc.com/burmese/articles/crwydv1nywepo
  saved: 53 paragraphs, 8 figures
Article: https://www.bbc.com/burmese/articles/c620regwn9j9o
  saved: 92 paragraphs, 6 figures
Article: https://www.bbc.com/burmese/articles/c6d09dn0vzgdo
  saved: 52 paragraphs, 11 figures
Article: https://www.bbc.com/burmese/articles/c1k234z87pdo
  saved: 59 paragraphs, 6 figures
Article: https://www.bbc.com/burmese/articles/cv5ynkpnzp7ro
  saved: 32 paragraphs, 4 figures
Article: https://www.bbc.com/burmese/articles/cxyvrnl30n7do
  saved: 64 paragraphs, 4 figures
Article: https://www.bbc.com/burmese/articles/c65yn0vlkp8po
  saved: 48 paragraphs, 5 figures
Article: https://www.bbc.com/burmese/articles/cw4g17k13415o
  saved: 58 paragraphs, 4 figures
Article: https://www.bbc.com/burmese/articles/cmwyvd7xyd5wo

## Step 7: Check the results

In [14]:
articles = pd.read_csv(ARTICLES_CSV)
figures = pd.read_csv(FIGURES_CSV) if FIGURES_CSV.exists() else pd.DataFrame()
print("Articles:", len(articles), "| Figures:", len(figures))
print("Missing titles:", articles["title_raw"].isna().sum())
print("Missing dates:", articles["published_at"].isna().sum())
print("Empty bodies:", (articles["paragraph_count"] == 0).sum())
if len(figures):
    print("Figures without caption:", figures["original_caption"].isna().sum())
articles[["bbc_id", "title_raw", "published_at", "paragraph_count", "image_count"]].head()

Articles: 26 | Figures: 129
Missing titles: 0
Missing dates: 0
Empty bodies: 0
Figures without caption: 14


,bbc_id,title_raw,published_at,paragraph_count,image_count
0,cvgy4xn5nleo,မြန်မာ့မြေရှားလုပ်ကွက်တွေထဲက ဖိအားပေးခိုင်းစေခ...,2026-10-26T11:34:32.697Z,73,5
1,c6gqwg79zq7do,ဘိုင်ယန်မြူးနစ်နဲ့ အင်္ဂလန်တိုက်စစ်မှူး ဟယ်ရီက...,2026-10-27T01:46:06.494Z,55,2
2,cx62y022y8mdo,ဆားဘီးယားကျောင်းသားလှုပ်ရှားမှု - Vučić ကို ဘ...,2026-10-26T15:11:39.864Z,35,3
3,cwp93m03nkdko,၁၉၆၂ စစ်ကောင်စီနဲ့ ၁၉၂၆ ဖွား မြန်မာနာမည်ကျော် ...,2026-10-24T14:54:43.980Z,47,21
4,cke8rq7eng6mo,ရုရှားစစ်သားစုဆောင်းရေးတပ်ထဲက လွတ်မြောက်လာတဲ့ ...,2026-10-24T11:15:33.610Z,44,4
